# Model Training & Development Notebook

Run these cells in order. Each cell builds on the previous one — if you
edit `src/` files, just re-run the relevant cell (Jupyter keeps state in
memory, so you don't need to re-run everything unless you change something
upstream).

**Kernel**: make sure this notebook is using your project's `.venv` Python
interpreter (VS Code will usually prompt you to select it, or use the
kernel picker in the top right).

In [ ]:
import sys
sys.path.insert(0, '..')  # so `src.` imports work from notebooks/ folder

import pandas as pd
import numpy as np

from src.utils.config import load_config
from src.data.load import load_raw_transactions
from src.data.clean import run_cleaning_pipeline
from src.data.validate import run_validation
from src.features.build_features import build_customer_feature_table
from src.features.preprocessing import stratified_split, get_feature_columns, fit_scaler, apply_scaler

pd.set_option('display.max_columns', None)

## 1. Load, validate, clean (skip if already cached)

In [ ]:
cfg = load_config('../config/config.yaml')
raw = load_raw_transactions(cfg['paths']['raw_data'], cfg['sheets'])
print('raw shape:', raw.shape)

result = run_validation(raw, cfg)
print('validation passed:', result.passed, result.errors)

cleaned = run_cleaning_pipeline(raw, cfg['cleaning']['non_product_stockcodes'], cfg['cleaning']['outlier_iqr_multiplier'])
cleaned.to_parquet('../data/interim/cleaned_transactions.parquet')
print('cleaned shape:', cleaned.shape, '| dupes removed:', cleaned.attrs['n_duplicates_removed'])

FileNotFoundError: Raw dataset not found at data/raw/online_retail_II.xlsx. Download it (UCI 'Online Retail II', id=502) and place it there.

## 2. Build the customer feature table

In [ ]:
cutoff = cleaned['invoice_date'].max() - pd.Timedelta(days=90)
feats = build_customer_feature_table(cleaned, cutoff, churn_window_days=90)
feats.to_parquet('../data/processed/customer_features.parquet')

print('feature table shape:', feats.shape)
print(feats['churned'].value_counts(normalize=True).round(3))
feats.head()

## 3. Quick sanity check — no infinities/NaNs slipped through

In [ ]:
numeric_cols = feats.select_dtypes('number').columns
n_inf = sum(np.isinf(feats[c]).sum() for c in numeric_cols)
n_nan = feats[numeric_cols].isna().sum().sum()
print('infinite values:', n_inf, '| NaN values:', n_nan)
assert n_inf == 0 and n_nan == 0, 'Fix build_features.py before continuing!'

## 4. Split and scale

In [ ]:
train, val, test = stratified_split(feats)
print('train:', train.shape, 'val:', val.shape, 'test:', test.shape)
print('churn rate - train:', train['churned'].mean().round(3),
      'val:', val['churned'].mean().round(3), 'test:', test['churned'].mean().round(3))

feature_cols = [c for c in get_feature_columns(feats, exclude=['customer_id', 'churned'])
                 if not c.startswith('affinity_')]
print('feature_cols:', feature_cols)

scaler = fit_scaler(train)
X_train_unscaled = train[feature_cols]
X_val_unscaled = val[feature_cols]
X_test_unscaled = test[feature_cols]
X_train_scaled = apply_scaler(X_train_unscaled, scaler)
X_val_scaled = apply_scaler(X_val_unscaled, scaler)
X_test_scaled = apply_scaler(X_test_unscaled, scaler)
y_train, y_val, y_test = train['churned'], val['churned'], test['churned']

## 5. Train all classical models

This runs cross-validated grid search across 6 models. Takes a few
minutes — grab a coffee.

In [ ]:
from src.models.train_classical import train_all_classical_models

fitted, log = train_all_classical_models(
    X_train_scaled, X_train_unscaled, y_train,
    X_test_scaled, X_test_unscaled, y_test,
    model_artifacts_dir='../model_artifacts',
    log_path='../data/processed/classical_model_comparison.csv',
)
log[['model', 'status', 'roc_auc', 'precision', 'recall', 'f1', 'train_time_sec']]

## 6. Visualize the comparison

In [ ]:
import matplotlib.pyplot as plt

trained = log[log['status'] == 'trained'].sort_values('roc_auc', ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(trained['model'], trained['roc_auc'])
ax.set_xlabel('ROC-AUC')
ax.set_title('Classical Model Comparison')
plt.tight_layout()
plt.show()